In [43]:
from typing import Iterator
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document as LCDocument
from docling.document_converter import DocumentConverter


In [44]:
class DoclingPDFLoader(BaseLoader):
    def __init__(self, file_path: str | list[str]) -> None:
        self._file_paths = file_path if isinstance(file_path, list) else [file_path]
        self._converter = DocumentConverter()


    def lazy_load(self) -> Iterator[LCDocument]:
        for source in self._file_paths:
            dl_doc = self._converter.convert(source).document
            text = dl_doc.export_to_markdown()
            yield LCDocument(page_content=text)

In [13]:
loader = DoclingPDFLoader(
    ['/Users/a/Programming/Langchain-Project/external-data/UAS Capstone Project.pdf', 
     '/Users/a/Programming/Langchain-Project/Raw-data/Chatbots-in-Academia-A-Retrieval-Augmented-Generation-Approach-for-Improved-Efficient-Information.pdf'
    ])

In [14]:
docs = loader.load()

2026-01-21 16:59:04,921 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-21 16:59:04,938 - INFO - Going to convert document batch...
2026-01-21 16:59:04,938 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-01-21 16:59:04,939 - INFO - Auto OCR model selected ocrmac.
2026-01-21 16:59:04,939 - INFO - Accelerator device: 'mps'
2026-01-21 16:59:06,567 - INFO - Accelerator device: 'mps'
2026-01-21 16:59:07,213 - INFO - Processing document UAS Capstone Project.pdf
2026-01-21 16:59:20,375 - INFO - Finished converting document UAS Capstone Project.pdf in 15.46 sec.
2026-01-21 16:59:20,414 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-21 16:59:20,415 - INFO - Going to convert document batch...
2026-01-21 16:59:20,416 - INFO - Processing document Chatbots-in-Academia-A-Retrieval-Augmented-Generation-Approach-for-Improved-Efficient-Information.pdf
2026-01-21 16:59:25,744 - INFO - Finished converting document

In [20]:
docs[1].page_content[:2000]

"## Chatbots in Academia: A Retrieval-Augmented Generation Approach for Improved Efficient Information Access\n\n## 1 st Maryamah Maryamah\n\nData Science Technology Faculty of Advanced Technology and Multidiscipline Universitas Airlangga Surabaya, Indonesia maryamah@ftmm.unair.ac.id\n\n4 th Netri Alia Rahmi\n\nData Science Technology Faculty of Advanced Technology and Multidiscipline Universitas Airlangga Surabaya, Indonesia netri.alia.rahmi-2021@ftmm.unair.ac.id\n\n2 nd Muhammad Maula Irfani Data Science Technology Faculty of Advanced Technology and Multidiscipline Universitas Airlangga Surabaya, Indonesia muhammad.maula.fani2021@ftmm.unair.ac.id\n\n## 5 th Mohammad Ghani\n\nData Science Technology Faculty of Advanced Technology and\n\nMultidiscipline\n\nUniversitas Airlangga\n\nSurabaya, Indonesia\n\nmohammad.ghani@ftmm.unair.ac.id\n\nAbstract -In today's digital age, higher education utilizes chatbots as virtual assistants to assist users, especially prospective students to access 

### chunking the documents

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

In [23]:
splits = text_splitter.split_documents(docs)

In [27]:
splits[130]

Document(metadata={}, page_content="In the BLEU Score evaluation, GPT-3.5-Turbo outperforms Gemini Pro, signifying better linguistic quality. The higher BLEU score reflects improved overlap with reference answers' n-grams, highlighting GPT-3.5-Turbo's superior text generation. Overall, GPT-3.5-Turbo emerges as a superior choice, aligning with ROUGE and BLEU metrics. Its ability to produce similar, high-quality text reinforces its efficacy in supporting chatbot responses for academic information, solidifying its role among Large Language Models.\n\n## V. CONCLUSION\n\nThis paper presents a comprehensive study on the development and evaluation of a chatbot for academic information utilizing the Retrieval-Augmented Generation (RAG) technique, specifically integrating OpenAI GPT-3.5 as the Large Language Model (LLM). The research explores the performance of the proposed chatbot against competitive methods such as Google Gemini, considering both retriever and language model aspects.")

### Retrieval process with BM25 --- Sparse Retrieval

In [11]:
from langchain_community.retrievers import BM25Retriever

In [29]:
retriever = BM25Retriever.from_documents(splits)
retriever.k = 3

In [32]:
retriever.invoke("siapa itu dek Nung yang selalu mewarnai hari")

[Document(metadata={}, page_content='1. Orang  tua penulis yang selalu memberikan dukungan moral dan materil kepada penulis. Terlebih terima kasih kepada donatur atau ayah penulis yang selalu sigap memberikan bantuan.\n2. Teman-teman kosan kahfi yang selalu ada dalam suka dalam sedih, teman penulis, Fatur  yang  selalu  bareng-bareng  belajar  dan  mengerjakan  skripsi.  Begitupun teman-teman yang lain.\n3. Bro  Cinere  (dek  Nung)  yang  selalu  menemani,  memberikan  warna  dan  selalu memberikan semangat bagi penulis.\n\nDalam penulisan laporan ini terdapat kekurangan-kekurangan yang mungkin disebabkan  oleh  keterbatasan  kemampuan  dan  pengetahuan  yang  penulis.  Namun, penulis berusaha menyelesaikan laporan ini sebaik mungkin. Oleh karena itu, apabila terdapat  kekurangan  di  dalam  penulisan  laporan  MBKM  ini,  dengan  rendah  hati penulis menerima kritik dan saran dari pembaca.'),
 Document(metadata={}, page_content='@cl. on\\_message async def on\\_nessage(message: cl.Mes

### Generated with Gemini Key

In [33]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=1.0,  # Gemini 3.0+ defaults to 1.0
    max_tokens=None,
    timeout=None,
    max_retries=2,
    
)

In [36]:
setup_and_retrieval = RunnableParallel(
    {
        "context":retriever,
        "question":RunnablePassthrough()
    }
)

In [37]:
prompt = ChatPromptTemplate.from_template(
    """Kamu adalah asisten cerdas. Jawablah pertanyaan pengguna berdasarkan konteks berikut ini saja.
    
    KONTEKS:
    {context}
    
    INSTRUKSI TAMBAHAN:
    1. Jawablah pertanyaan berdasarkan KONTEKS di atas.
    2. Kamu WAJIB menjawab dalam Bahasa Indonesia, terlepas dari bahasa apa pertanyaan itu diberikan.
    3. Jika jawaban tidak ada di dalam konteks, katakan "Maaf, informasi tidak ditemukan dalam dokumen."
    
    PERTANYAAN: {question}
    JAWABAN:"""
)

In [39]:
chain_gemini = setup_and_retrieval | prompt | model | StrOutputParser()

In [42]:
chain_gemini.invoke("siapa teman penulis yang selalu menugas bareng")

2026-01-21 17:18:40,245 - INFO - AFC is enabled with max remote calls: 10.
2026-01-21 17:18:42,701 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"


'Teman penulis yang selalu belajar dan mengerjakan skripsi bersama-sama adalah Fatur.'

### Menggunakan model Local

In [45]:
from langchain_community.chat_models import ChatOllama

In [46]:
llm_model = ChatOllama(
    model='mistral-openorca:7b-q4_0',
    temperature=0,
    streaming=True
)

/var/folders/05/16tm42991snf6c27rs7nssdh0000gn/T/ipykernel_68493/3972573145.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm_model = ChatOllama(


In [47]:
generative_local = setup_and_retrieval | prompt | llm_model | StrOutputParser()

In [48]:
generative_local.invoke("siapa teman penulis yang selalu menugas bareng")

' Teman-teman kosan kahfi yang selalu ada dalam suka dan sedih, teman penulis, Fatur yang selalu bareng-bareng belajar dan mengerjakan skripsi. Begitupun teman-teman yang lain.'